# Neste notebook é realizado um extract do dados abertos da PRF (Datatran)

In [ ]:
import gdown as gd
import pandas as pd
from pathlib import Path

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yaml").exists())
data_dir = raiz / "data"
data_dir.mkdir(exist_ok=True)

In [46]:
url = {
    "datatran2025": "https://drive.google.com/file/d/1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR/view",
    "datatran2024": "https://drive.google.com/file/d/14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO/view",
    "datatran2023": "https://drive.google.com/file/d/1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq/view"
}

for nome, link in url.items():
    output = data_dir / f"{nome}.zip"
    gd.download(link, str(output))

Downloading...
From: https://drive.google.com/uc?id=1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR
To: c:\projeto-sinistros-prf\data\datatran2025.zip
100%|██████████| 3.73M/3.73M [00:00<00:00, 5.90MB/s]
Downloading...
From: https://drive.google.com/uc?id=14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO
To: c:\projeto-sinistros-prf\data\datatran2024.zip
100%|██████████| 3.78M/3.78M [00:00<00:00, 6.00MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq
To: c:\projeto-sinistros-prf\data\datatran2023.zip
100%|██████████| 3.40M/3.40M [00:00<00:00, 5.94MB/s]


In [ ]:
# Shape de cada arquivo antes de concatenar
total_linhas = 0

for file in sorted(data_dir.glob("datatran*.zip")):
    data = pd.read_csv(file, sep=";", encoding="latin-1", dtype=str)
    print(file.name, data.shape)
    total_linhas += data.shape[0]

print("total:", total_linhas)

In [ ]:
# Leitura dos 3 arquivos (tudo como texto) + metadados de origem, como na bronze
dfs = []
for file in sorted(data_dir.glob("datatran*.zip")):
    data = pd.read_csv(file, sep=";", encoding="latin-1", dtype=str)
    data["_ano_arquivo"] = file.stem.removeprefix("datatran")
    data["_arquivo_origem"] = file.name
    dfs.append(data)

df = pd.concat(dfs, ignore_index=True)
df.shape

In [56]:
# Tipos de dados
df.dtypes

id                        str
data_inversa              str
dia_semana                str
horario                   str
uf                        str
br                        str
km                        str
municipio                 str
causa_acidente            str
tipo_acidente             str
classificacao_acidente    str
fase_dia                  str
sentido_via               str
condicao_metereologica    str
tipo_pista                str
tracado_via               str
uso_solo                  str
pessoas                   str
mortos                    str
feridos_leves             str
feridos_graves            str
ilesos                    str
ignorados                 str
feridos                   str
veiculos                  str
latitude                  str
longitude                 str
regional                  str
delegacia                 str
uop                       str
dtype: object

In [67]:
df.head()

,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,tipo_acidente,classificacao_acidente,fase_dia,sentido_via,condicao_metereologica,tipo_pista,tracado_via,uso_solo,pessoas,mortos,feridos_leves,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,496519,2023-01-01,domingo,02:00:00,ES,101,114,SOORETAMA,Ausência de reação do condutor,Saída de leito carroçável,Com Vítimas Feridas,Plena Noite,Crescente,Céu Claro,Simples,Reta,Não,1,0,1,0,0,0,1,1,"-19,09484877","-40,05095848",SPRF-ES,DEL04-ES,UOP01-DEL04-ES
1,496543,2023-01-01,domingo,03:40:00,SP,116,"113,1",TAUBATE,Entrada inopinada do pedestre,Atropelamento de Pedestre,Com Vítimas Fatais,Plena Noite,Decrescente,Céu Claro,Dupla,Reta,Sim,5,1,0,0,0,4,0,2,"-23,0445658","-45,58259814",SPRF-SP,DEL02-SP,UOP02-DEL02-SP
2,496590,2023-01-01,domingo,01:40:00,MT,163,1112,GUARANTA DO NORTE,Reação tardia ou ineficiente do condutor,Tombamento,NaN,Plena Noite,Crescente,Ignorado,Simples,Curva;Declive,Não,2,0,0,1,0,2,1,3,"-9,70020602","-54,87588757",SPRF-MT,DEL06-MT,UOP03-DEL06-MT
3,496610,2023-01-01,domingo,10:40:00,PR,376,"314,8",ORTIGUEIRA,Velocidade Incompatível,Tombamento,Sem Vítimas,Pleno dia,Crescente,Sol,Dupla,Curva,Não,2,0,0,0,1,2,0,3,"-23,985512","-51,083555",SPRF-PR,DEL07-PR,UOP02-DEL07-PR
4,496659,2023-01-01,domingo,14:55:00,MG,116,"569,4",MANHUACU,Acumulo de água sobre o pavimento,Colisão frontal,Com Vítimas Feridas,Pleno dia,Decrescente,Chuva,Simples,Declive;Curva,Não,4,0,0,2,1,1,2,3,"-20,10007457","-42,17884091",SPRF-MG,DEL06-MG,UOP03-DEL06-MG


In [49]:
# Quantidade de valores ausentes por coluna
df.isnull().sum()

id                         0
data_inversa               0
dia_semana                 0
horario                    0
uf                         0
br                         0
km                         0
municipio                  0
causa_acidente             0
tipo_acidente              0
classificacao_acidente     1
fase_dia                   0
sentido_via                0
condicao_metereologica     0
tipo_pista                 0
tracado_via                0
uso_solo                   0
pessoas                    0
mortos                     0
feridos_leves              0
feridos_graves             0
ilesos                     0
ignorados                  0
feridos                    0
veiculos                   0
latitude                   0
longitude                  0
regional                   2
delegacia                 22
uop                       38
dtype: int64

In [ ]:
# Quantidade de id duplicados
df.duplicated("id").sum()

np.int64(0)

In [ ]:
# Os 3 anos têm as mesmas colunas, na mesma ordem?
referencia = list(dfs[0].columns)
{d["_ano_arquivo"].iloc[0]: list(d.columns) == referencia for d in dfs}

In [ ]:
# Nulos "disfarçados" de texto, que o isnull() não enxerga
padrao = r"(?i)ignorad|n[aã]o informad|null|^\s*$"

for col in df.columns.drop(["_ano_arquivo", "_arquivo_origem"]):
    achados = df[col].dropna()
    achados = achados[achados.str.contains(padrao)]
    if not achados.empty:
        print(col, achados.value_counts().to_dict())

In [ ]:
# Os nulos de classificacao_acidente podem ser deduzidos pelas contagens?
df[df["classificacao_acidente"].isna()][["id", "_ano_arquivo", "mortos", "feridos_leves", "feridos_graves"]]

In [ ]:
# Cobertura de datas por arquivo: ano completo e sem datas de outro ano?
df.groupby("_ano_arquivo").agg(
    data_min=("data_inversa", "min"),
    data_max=("data_inversa", "max"),
    meses=("data_inversa", lambda s: s.str[:7].nunique()),
    linhas=("id", "size"),
)

In [ ]:
# Formato consistente nos 3 anos? (AAAA-MM-DD e HH:MM:SS)
df.groupby("_ano_arquivo").agg(
    data_ok=("data_inversa", lambda s: s.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()),
    horario_ok=("horario", lambda s: s.str.fullmatch(r"\d{2}:\d{2}:\d{2}").all()),
)

## Conclusões da exploração

**Parâmetros de leitura (iguais para os 3 anos)**
- Arquivo: zip com um único CSV (`datatranAAAA.csv`), lido direto do zip
- `sep=";"`, `encoding="latin-1"`, `dtype=str` (nada é convertido na bronze)

**Volume e schema**
- 2023: 67.766 linhas · 2024: 73.202 · 2025: 72.529 · **total: 213.497**
- 30 colunas, **mesmos nomes e mesma ordem nos 3 anos**, sem drift de schema
- `id` é **único inclusive entre anos**, então serve como chave da tabela

**Datas e horários**
- Cada arquivo cobre o ano completo (01/01 a 31/12, 12 meses), sem datas de outro ano
- `data_inversa` sempre `AAAA-MM-DD` e `horario` sempre `HH:MM:SS` nos 3 anos

**Qualidade, ou seja, o que fica para a silver**
- `classificacao_acidente`: 3 nulos (1 por ano), **deduzíveis** pelas contagens (`mortos` > 0 → "Com Vítimas Fatais"; feridos > 0 → "Com Vítimas Feridas")
- Nulos disfarçados de texto: `condicao_metereologica = "Ignorado"` (2.787) e `sentido_via = "Não Informado"` (516) → padronizar para nulo
- `km`, `latitude` e `longitude` usam vírgula decimal (`km` às vezes sem casas decimais) → converter para numérico
- `tracado_via` tem vários valores numa célula, em ordem variável ("Curva;Declive" = "Declive;Curva"), em 25% das linhas
- Contagens (`mortos`, `feridos`...) e `br` → converter para inteiro
- `regional`, `delegacia` e `uop` têm poucos nulos (2, 22, 38): colunas administrativas, sem impacto na análise

**Schema da bronze:** as 30 colunas originais como `STRING` + `_ano_arquivo`, `_arquivo_origem` e `_ingested_at`